# Binary Encoder

In [1]:
import pandas as pd

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings("ignore")

# Create sample dataset with high cardinality
np.random.seed(42)


In [3]:
n_samples = 1000

# Simulate high-cardinality features
cities = [f'City_{i}' for i in range(100)]  # 100 unique cities
products = [f'Product_{i}' for i in range(50)]  # 50 unique products

data = pd.DataFrame({
    'city': np.random.choice(cities, n_samples),
    'product': np.random.choice(products, n_samples),
    'age': np.random.randint(18, 70, n_samples),
    'purchased': np.random.randint(0, 2, n_samples)
})

print("Binary Encoding Example")
print("="*60)
print(f"\nDataset shape: {data.shape}")
print(f"Unique cities: {data['city'].nunique()}")
print(f"Unique products: {data['product'].nunique()}")


Binary Encoding Example

Dataset shape: (1000, 4)
Unique cities: 100
Unique products: 50


In [13]:
import pandas as pd

def binary_encode(series):
    """
    Perform binary encoding on a categorical series.
    
    Parameters:
    -----------
    series : pd.Series
        Categorical series to encode
    
    Returns:
    --------
    tuple: (pd.DataFrame, dict) Binary-encoded features and the mapping
    """
    # Get unique categories and create mapping
    categories = sorted(series.unique())
    cat_to_int = {cat: i for i, cat in enumerate(categories)}
    
    # Calculate number of bits needed
    max_int = len(categories) - 1
    n_bits = max_int.bit_length()
    
    # Convert categories to integers
    integers = series.map(cat_to_int)
    
    # Convert integers to binary
    binary_df = pd.DataFrame(index=series.index)
    
    for bit in range(n_bits):
        # Extract each bit using .to_numpy() to bypass the Series limitation
        binary_df[f'bin_{bit}'] = (integers.to_numpy() >> bit) & 1
    
    return binary_df, cat_to_int

### Method 1: Manual Binary Encoding

In [14]:
print("\n" + "="*60)
print("Method 1: Manual Binary Encoding")
print("="*60)

# Apply binary encoding
city_binary, city_mapping = binary_encode(data['city'])

print(f"\nBinary encoding for 'city' feature:")
print(f"  Original categories: {data['city'].nunique()}")
print(f"  Binary columns created: {city_binary.shape[1]}")
print(f"  Dimensionality reduction: {data['city'].nunique()} → {city_binary.shape[1]}")
print(f"  Space saving: {(1 - city_binary.shape[1]/data['city'].nunique())*100:.1f}%")

# Show example encodings

print("\nSample binary encodings:")
sample_cities = sorted(data['city'].unique())[:8]
for city in sample_cities:
    city_int = city_mapping[city]
    binary_str = format(city_int, f'0{city_binary.shape[1]}b')
    row = data[data['city'] == city].index[0]
    encoding = city_binary.loc[row].values
    encoding_str = ' '.join(map(str, encoding))  # Convert array to string
    print(f"  {city:15s} → {city_int:3d} → {binary_str} → [{encoding_str}]")



Method 1: Manual Binary Encoding

Binary encoding for 'city' feature:
  Original categories: 100
  Binary columns created: 7
  Dimensionality reduction: 100 → 7
  Space saving: 93.0%

Sample binary encodings:
  City_0          →   0 → 0000000 → [0 0 0 0 0 0 0]
  City_1          →   1 → 0000001 → [1 0 0 0 0 0 0]
  City_10         →   2 → 0000010 → [0 1 0 0 0 0 0]
  City_11         →   3 → 0000011 → [1 1 0 0 0 0 0]
  City_12         →   4 → 0000100 → [0 0 1 0 0 0 0]
  City_13         →   5 → 0000101 → [1 0 1 0 0 0 0]
  City_14         →   6 → 0000110 → [0 1 1 0 0 0 0]
  City_15         →   7 → 0000111 → [1 1 1 0 0 0 0]


### Method 2: Comparison of Dimensionality

In [15]:
print("\n" + "="*60)
print("Method 2: Dimensionality Comparison")
print("="*60)

def compare_encoding_dimensions(n_categories):
    """Compare number of columns for different encodings"""
    onehot_cols = n_categories
    binary_cols = (n_categories - 1).bit_length()
    label_cols = 1
    
    return {
        'Categories': n_categories,
        'One-Hot': onehot_cols,
        'Binary': binary_cols,
        'Label': label_cols,
        'Binary_Reduction': f"{(1 - binary_cols/onehot_cols)*100:.1f}%"
    }

# Compare different cardinalities
cardinalities = [10, 50, 100, 500, 1000, 5000]
comparison_data = [compare_encoding_dimensions(n) for n in cardinalities]
comparison_df = pd.DataFrame(comparison_data)

print("\nEncoding Dimensionality Comparison:")
print(comparison_df.to_string(index=False))

print("\n💡 Binary encoding provides exponential compression!")
print("   Formula: binary_columns = ceil(log2(n_categories))")


Method 2: Dimensionality Comparison

Encoding Dimensionality Comparison:
 Categories  One-Hot  Binary  Label Binary_Reduction
         10       10       4      1            60.0%
         50       50       6      1            88.0%
        100      100       7      1            93.0%
        500      500       9      1            98.2%
       1000     1000      10      1            99.0%
       5000     5000      13      1            99.7%

💡 Binary encoding provides exponential compression!
   Formula: binary_columns = ceil(log2(n_categories))


### Method 3: Model Performance Comparison

In [16]:
print("\n" + "="*60)
print("Method 3: Model Performance with Different Encodings")
print("="*60)

# Split data
train_data, test_data = train_test_split(data, test_size=0.25, random_state=42)

# Prepare target
y_train = train_data['purchased']
y_test = test_data['purchased']


Method 3: Model Performance with Different Encodings


#### Strategy 1: Binary Encoding

In [18]:
# Strategy 1: Binary Encoding
city_binary_train, city_map = binary_encode(train_data['city'])
product_binary_train, product_map = binary_encode(train_data['product'])

# Apply same encoding to test (handle unseen categories)
def apply_binary_encoding(series, mapping, n_bits):
    """Apply learned binary encoding to new data"""
    # Map categories, unseen get max+1
    max_val = max(mapping.values())
    integers = series.map(mapping).fillna(max_val + 1).astype(int)
    
    binary_df = pd.DataFrame(index=series.index)
    for bit in range(n_bits):
        binary_df[f'bin_{bit}'] = (integers.to_numpy() >> bit) & 1
    
    return binary_df

city_binary_test = apply_binary_encoding(test_data['city'], city_map, city_binary_train.shape[1])
product_binary_test = apply_binary_encoding(test_data['product'], product_map, product_binary_train.shape[1])

X_train_binary = pd.concat([
    city_binary_train.reset_index(drop=True),
    product_binary_train.reset_index(drop=True),
    train_data[['age']].reset_index(drop=True)
], axis=1)

X_test_binary = pd.concat([
    city_binary_test.reset_index(drop=True),
    product_binary_test.reset_index(drop=True),
    test_data[['age']].reset_index(drop=True)
], axis=1)

#### Strategy 2: One-Hot Encoding (for comparison)

In [19]:
train_onehot = pd.get_dummies(train_data[['city', 'product']], prefix=['city', 'product'])
test_onehot = pd.get_dummies(test_data[['city', 'product']], prefix=['city', 'product'])

# Align columns
test_onehot = test_onehot.reindex(columns=train_onehot.columns, fill_value=0)

X_train_onehot = pd.concat([
    train_onehot.reset_index(drop=True),
    train_data[['age']].reset_index(drop=True)
], axis=1)

X_test_onehot = pd.concat([
    test_onehot.reset_index(drop=True),
    test_data[['age']].reset_index(drop=True)
], axis=1)

In [20]:
# Train models
rf_binary = RandomForestClassifier(n_estimators=100, random_state=42)
rf_binary.fit(X_train_binary, y_train)

rf_onehot = RandomForestClassifier(n_estimators=100, random_state=42)
rf_onehot.fit(X_train_onehot, y_train)

# Evaluate
binary_acc = accuracy_score(y_test, rf_binary.predict(X_test_binary))
onehot_acc = accuracy_score(y_test, rf_onehot.predict(X_test_onehot))

In [21]:
print("\nRandom Forest Performance:")
print(f"\nBinary Encoding:")
print(f"  Features: {X_train_binary.shape[1]}")
print(f"  Training Accuracy: {rf_binary.score(X_train_binary, y_train):.4f}")
print(f"  Testing Accuracy: {binary_acc:.4f}")


Random Forest Performance:

Binary Encoding:
  Features: 14
  Training Accuracy: 1.0000
  Testing Accuracy: 0.5000


In [22]:
print(f"\nOne-Hot Encoding:")
print(f"  Features: {X_train_onehot.shape[1]}")
print(f"  Training Accuracy: {rf_onehot.score(X_train_onehot, y_train):.4f}")
print(f"  Testing Accuracy: {onehot_acc:.4f}")


One-Hot Encoding:
  Features: 151
  Training Accuracy: 1.0000
  Testing Accuracy: 0.4640


In [23]:
print(f"\nFeature reduction: {X_train_onehot.shape[1]} → {X_train_binary.shape[1]}")
print(f"Space saving: {(1 - X_train_binary.shape[1]/X_train_onehot.shape[1])*100:.1f}%")


Feature reduction: 151 → 14
Space saving: 90.7%


### Method 4: Using category_encoders library

In [24]:
print("\n" + "="*60)
print("Method 4: Using category_encoders Library")
print("="*60)

try:
    from category_encoders import BinaryEncoder
    
    # BinaryEncoder from the library
    binary_encoder = BinaryEncoder(cols=['city', 'product'])
    
    train_encoded = binary_encoder.fit_transform(train_data[['city', 'product']])
    test_encoded = binary_encoder.transform(test_data[['city', 'product']])
    
    print("\nBinaryEncoder output:")
    print(f"  Columns created: {train_encoded.shape[1]}")
    print(f"\nFirst few rows:")
    print(train_encoded.head())
    
    print(f"\n✅ category_encoders provides production-ready binary encoding")
    print(f"   Install: pip install category-encoders")
except ImportError:
    print("\n⚠️ Install category_encoders: pip install category-encoders")
    print("   This library provides optimized binary encoding")


Method 4: Using category_encoders Library

BinaryEncoder output:
  Columns created: 13

First few rows:
     city_0  city_1  city_2  city_3  city_4  city_5  city_6  product_0  \
82        0       0       0       0       0       0       1          0   
991       0       0       0       0       0       1       0          0   
789       0       0       0       0       0       1       1          0   
894       0       0       0       0       1       0       0          0   
398       0       0       0       0       1       0       1          0   

     product_1  product_2  product_3  product_4  product_5  
82           0          0          0          0          1  
991          0          0          0          1          0  
789          0          0          0          1          1  
894          0          0          1          0          0  
398          0          0          1          0          1  

✅ category_encoders provides production-ready binary encoding
   Install: pip insta

### Visualizing Binary Patterns

In [25]:
print("\n" + "="*60)
print("Method 5: Understanding Binary Patterns")
print("="*60)

# Show how binary encoding works for small example
small_categories = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
small_series = pd.Series(small_categories)
binary_small, _ = binary_encode(small_series)

print("\nBinary encoding for 8 categories:")
print("\nCategory | Int | Binary | Bit_0 Bit_1 Bit_2")
print("-" * 45)
for i, cat in enumerate(small_categories):
    binary_str = format(i, '03b')
    bits = binary_small.iloc[i].values
    print(f"   {cat}     |  {i}  |  {binary_str}   |   {int(bits[0])}     {int(bits[1])}     {int(bits[2])}")

print("\n💡 Each bit position can be thought of as a binary feature")
print("   Tree models can learn: 'If bit_2=1 AND bit_0=0 then...'")


Method 5: Understanding Binary Patterns

Binary encoding for 8 categories:

Category | Int | Binary | Bit_0 Bit_1 Bit_2
---------------------------------------------
   A     |  0  |  000   |   0     0     0
   B     |  1  |  001   |   1     0     0
   C     |  2  |  010   |   0     1     0
   D     |  3  |  011   |   1     1     0
   E     |  4  |  100   |   0     0     1
   F     |  5  |  101   |   1     0     1
   G     |  6  |  110   |   0     1     1
   H     |  7  |  111   |   1     1     1

💡 Each bit position can be thought of as a binary feature
   Tree models can learn: 'If bit_2=1 AND bit_0=0 then...'
